## 3. Перевірка якості та очищення даних (Data Quality & Cleaning)

In [20]:
# Імпортуємо необхідні бібліотеки
import pandas as pd
from pathlib import Path

In [21]:
# Знаходимо папку з вихідними датасетами

current_dir = Path.cwd()

possible_paths = [
    parent / "data" / "simulated"
    for parent in [current_dir, *current_dir.parents]
]

data_path = next(
    (path for path in possible_paths if path.is_dir()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Не вдалося знайти папку data/simulated"
    )

# Завантажуємо CSV-файли

csv_files = sorted(data_path.glob("*.csv"))

if len(csv_files) != 13:
    raise ValueError(
        f"Очікувалося 13 датасетів, знайдено {len(csv_files)}"
    )

datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

# Перевіряємо результат завантаження

original_rows = sum(
    len(df)
    for df in datasets.values()
)

print(f"Папка з даними: {data_path}")
print(f"Кількість датасетів: {len(datasets)}")
print(f"Загальна кількість записів: {original_rows}")

Папка з даними: c:\Users\ACER\github\analytics\data\simulated
Кількість датасетів: 13
Загальна кількість записів: 69431


In [23]:
# Створюємо копії вихідних датасетів

cleaned_datasets = {
    name: df.copy(deep=True)
    for name, df in datasets.items()
}

# Перевіряємо кількість записів

cleaned_rows = sum(
    len(df)
    for df in cleaned_datasets.values()
)

print(f"Кількість датасетів: {len(cleaned_datasets)}")
print(f"Записів в оригінальних даних: {original_rows}")
print(f"Записів у робочих копіях: {cleaned_rows}")

assert original_rows == cleaned_rows

print("Робочі копії успішно створено!")

Кількість датасетів: 13
Записів в оригінальних даних: 69431
Записів у робочих копіях: 69431
Робочі копії успішно створено!


In [24]:
# Визначаємо колонки за типами даних

id_columns = [
    "user_id",
    "participant_id",
    "room_id",
    "archive_session_id",
    "title_id",
    "room_title_id"
]

integer_columns = [
    "max_duration_minutes",
    "participant_count",
    "card_position",
    "release_year",
    "runtime_minutes",
    "likes_count"
]

float_columns = [
    "vote_average"
]

timestamp_columns = [
    "event_timestamp",
    "started_at",
    "completed_at"
]

string_columns = [
    "event_id",
    "page_view_id",
    "event_name",
    "event_value_type",
    "capture_side",
    "visitor_id",
    "user_type",
    "content_type",
    "interaction_method",
    "title",
    "genres_json"
]

# Створюємо список результатів перетворення

conversion_results = []

# Виконуємо перетворення типів даних

for name, df in cleaned_datasets.items():

    # Ідентифікатори та цілочисельні поля

    for column in id_columns + integer_columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="raise"
            ).astype("Int64")

            conversion_results.append({
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype)
            })

    # Числові поля з десятковими значеннями

    for column in float_columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="raise"
            ).astype("Float64")

            conversion_results.append({
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype)
            })

    # Часові мітки

    for column in timestamp_columns:

        if column in df.columns:

            df[column] = pd.to_datetime(
                df[column],
                errors="raise",
                utc=True
            )

            conversion_results.append({
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype)
            })

    # Логічне поле is_liked

    if "is_liked" in df.columns:

        normalized_values = (
            df["is_liked"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        invalid_mask = (
            normalized_values.notna()
            & ~normalized_values.isin(["true", "false"])
        )

        if invalid_mask.any():

            invalid_values = (
                normalized_values[invalid_mask]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"Некоректні значення is_liked у датасеті "
                f"{name}: {invalid_values}"
            )

        df["is_liked"] = normalized_values.map({
            "true": True,
            "false": False
        }).astype("boolean")

        conversion_results.append({
            "dataset": name,
            "column": "is_liked",
            "dtype": str(df["is_liked"].dtype)
        })

    # Текстові поля

    for column in string_columns:

        if column in df.columns:

            df[column] = df[column].astype("string")

            conversion_results.append({
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype)
            })

# Формуємо звіт

conversion_summary = pd.DataFrame(conversion_results)

display(conversion_summary)

print(f"Виконано перетворень: {len(conversion_summary)}")

,dataset,column,dtype
0,01_landing_page,user_id,Int64
1,01_landing_page,participant_id,Int64
2,01_landing_page,room_id,Int64
3,01_landing_page,archive_session_id,Int64
4,01_landing_page,title_id,Int64
...,...,...,...
359,12_settings,user_type,string
360,12_settings,content_type,string
361,12_settings,interaction_method,string
362,12_settings,title,string


Виконано перетворень: 364


In [25]:
# Визначаємо неправильну та правильну назви події

old_event_name = "sign_in_from_signup_clicked"
new_event_name = "sign_in_from_sign_up_clicked"

rename_results = []
renamed_total = 0

# Замінюємо назву події у всіх датасетах

for name, df in cleaned_datasets.items():

    if "event_name" in df.columns:

        records_to_rename = (
            df["event_name"] == old_event_name
        ).sum()

        if records_to_rename > 0:

            df["event_name"] = df["event_name"].replace(
                old_event_name,
                new_event_name
            )

            renamed_total += records_to_rename

            rename_results.append({
                "dataset": name,
                "renamed_records": records_to_rename
            })

# Формуємо звіт

rename_summary = pd.DataFrame(rename_results)

display(rename_summary)

# Перевіряємо результат

old_count = sum(
    (df["event_name"] == old_event_name).sum()
    for df in cleaned_datasets.values()
    if "event_name" in df.columns
)

print(f"Виправлено записів: {renamed_total}")
print(f"Записів зі старою назвою: {old_count}")

assert old_count == 0
assert renamed_total == 212

print("Перевірку перейменування успішно пройдено!")

,dataset,renamed_records
0,10_sign_up,212


Виправлено записів: 212
Записів зі старою назвою: 0
Перевірку перейменування успішно пройдено!


In [26]:
# Визначаємо очікувані типи даних

expected_dtypes = {
    "user_id": "Int64",
    "participant_id": "Int64",
    "room_id": "Int64",
    "archive_session_id": "Int64",
    "title_id": "Int64",
    "room_title_id": "Int64",

    "max_duration_minutes": "Int64",
    "participant_count": "Int64",
    "card_position": "Int64",
    "release_year": "Int64",
    "runtime_minutes": "Int64",
    "likes_count": "Int64",

    "vote_average": "Float64",
    "is_liked": "boolean",

    "event_timestamp": "datetime64[ns, UTC]",
    "started_at": "datetime64[ns, UTC]",
    "completed_at": "datetime64[ns, UTC]",

    "event_id": "string",
    "page_view_id": "string",
    "event_name": "string",
    "event_value_type": "string",
    "capture_side": "string",
    "visitor_id": "string",
    "user_type": "string",
    "content_type": "string",
    "interaction_method": "string",
    "title": "string",
    "genres_json": "string"
}

# Формуємо підсумковий звіт

validation_results = []
dtype_issues_details = []

for name, df in cleaned_datasets.items():

    original_df = datasets[name]

    # Перевіряємо кількість рядків та структуру колонок

    rows_match = len(df) == len(original_df)
    columns_match = list(df.columns) == list(original_df.columns)

    # Перевіряємо типи даних

    dtype_issues = []

    for column, expected_dtype in expected_dtypes.items():

        if column in df.columns:

            actual_dtype = str(df[column].dtype)

            if actual_dtype != expected_dtype:

                dtype_issues.append(column)

                dtype_issues_details.append({
                    "dataset": name,
                    "column": column,
                    "actual_dtype": actual_dtype,
                    "expected_dtype": expected_dtype
                })

    # Перевіряємо стару назву події

    old_event_count = (
        (df["event_name"] == old_event_name).sum()
        if "event_name" in df.columns
        else 0
    )

    # Визначаємо статус

    status = (
        "OK"
        if rows_match
        and columns_match
        and len(dtype_issues) == 0
        and old_event_count == 0
        else "CHECK"
    )

    validation_results.append({
        "dataset": name,
        "rows_before": len(original_df),
        "rows_after": len(df),
        "rows_match": rows_match,
        "columns_match": columns_match,
        "dtype_issues": len(dtype_issues),
        "old_event_count": old_event_count,
        "status": status
    })

# Формуємо таблицю перевірки

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

# Перевіряємо загальну кількість записів

final_rows = sum(
    len(df)
    for df in cleaned_datasets.values()
)

assert final_rows == original_rows, (
    "Загальна кількість записів змінилася"
)

# Перевіряємо результати всіх датасетів

assert validation_summary["status"].eq("OK").all(), (
    "Не всі датасети пройшли перевірку. "
    "Перегляньте validation_summary."
)

print("Усі датасети успішно пройшли фінальну перевірку!")
print(f"Кількість датасетів: {len(cleaned_datasets)}")
print(f"Загальна кількість записів: {final_rows}")

,dataset,rows_before,rows_after,rows_match,columns_match,dtype_issues,old_event_count,status
0,01_landing_page,15000,15000,True,True,0,0,OK
1,02_authenticated_create_room,900,900,True,True,0,0,OK
2,02_guest_create_room,1375,1375,True,True,0,0,OK
3,03_authenticated_host_room_invite_friends,900,900,True,True,0,0,OK
4,03_guest_host_room_invite_friends,1100,1100,True,True,0,0,OK
5,06_authenticated_swipe_session,16170,16170,True,True,0,0,OK
6,06_guest_swipe_session,21230,21230,True,True,0,0,OK
7,08_authenticated_results_winners,3813,3813,True,True,0,0,OK
8,08_guest_results_winners,3903,3903,True,True,0,0,OK
9,09_sign_in,960,960,True,True,0,0,OK


Усі датасети успішно пройшли фінальну перевірку!
Кількість датасетів: 13
Загальна кількість записів: 69431


In [27]:
# Зберігаємо очищені датасети у папку data_processed

# Визначаємо шлях до папки з обробленими даними

project_root = data_path.parent.parent

processed_path = (
    project_root
    / "test_analytics"
    / "data_processed"
)

# Створюємо папку, якщо вона ще не існує

processed_path.mkdir(parents=True, exist_ok=True)

# Зберігаємо кожен датасет у форматі CSV

for name, df in cleaned_datasets.items():

    output_file = processed_path / f"{name}.csv"

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

# Перевіряємо збережені файли

saved_files = sorted(processed_path.glob("*.csv"))

assert len(saved_files) == len(cleaned_datasets), (
    "Кількість збережених файлів не відповідає кількості датасетів"
)

# Перевіряємо, чи збережено правильну кількість записів

saved_rows = sum(
    len(pd.read_csv(file))
    for file in saved_files
)

assert saved_rows == original_rows, (
    "Кількість записів у збережених файлах не відповідає оригіналу"
)

print(f"Папка з очищеними даними: {processed_path}")
print(f"Кількість збережених файлів: {len(saved_files)}")
print(f"Загальна кількість записів: {saved_rows}")

for file in saved_files:
    print(file.name)

print("Усі очищені датасети успішно збережено!")

Папка з очищеними даними: c:\Users\ACER\github\analytics\test_analytics\data_processed
Кількість збережених файлів: 13
Загальна кількість записів: 69431
01_landing_page.csv
02_authenticated_create_room.csv
02_guest_create_room.csv
03_authenticated_host_room_invite_friends.csv
03_guest_host_room_invite_friends.csv
06_authenticated_swipe_session.csv
06_guest_swipe_session.csv
08_authenticated_results_winners.csv
08_guest_results_winners.csv
09_sign_in.csv
10_sign_up.csv
11_home_signed_in.csv
12_settings.csv
Усі очищені датасети успішно збережено!




На етапі очищення та підготовки даних було виконано стандартизацію типів даних і нормалізацію окремих значень у 13 датасетах проєкту Movie Match.

**Основні результати:**

- Створено окремі робочі копії вихідних датасетів, що дозволило зберегти оригінальні дані без змін.
- Ідентифікатори приведено до типу `Int64`.
- Числові поля приведено до типів `Int64` та `Float64`.
- Часові мітки перетворено на `datetime64[ns, UTC]`.
- Логічне поле `is_liked` приведено до типу `boolean`.
- Текстові поля приведено до типу `string`.
- Виправлено назву події `sign_in_from_signup_clicked` на `sign_in_from_sign_up_clicked` у 212 записах.
- Структурні пропущені значення збережено без автоматичного заповнення або видалення.
- Дублікати на цьому етапі не видалялися, оскільки попередні перевірки не виявили порушень унікальності `event_id` та повторних голосів за визначеною комбінацією ідентифікаторів.

За результатами фінальної перевірки всі 13 датасетів отримали статус `OK`.

Загальний обсяг даних залишився незмінним — 69 431 запис. Структуру таблиць збережено.

Очищені та стандартизовані дані доступні в об'єкті `cleaned_datasets` та готові до подальшого дослідницького аналізу.